In [1]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     - -------------------------------------- 0.5/12.8 MB 9.0 MB/s eta 0:00:02
     --------- ------------------------------ 3.1/12.8 MB 12.1 MB/s eta 0:00:01
     ------------------ --------------------- 5.8/12.8 MB 12.4 MB/s eta 0:00:01
     ------------------------ --------------- 7.9/12.8 MB 12.0 MB/s eta 0:00:01
     ------------------------------- ------- 10.5/12.8 MB 12.0 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 12.0 MB/s  0:00:01
[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
!python ../preprocessing.py

Loading data...
Data quality check:
Shape: (27131, 12)
Total Papers: 27131
Total Columns: 12

Missing values per column:
id                 0
submitter          0
authors            0
title              0
comments       10158
journal-ref    24190
doi            24643
abstract           0
report-no      25780
categories         0
versions           0
year               0
dtype: int64

Duplicate IDs: 0

Duplicate Abstracts: 5

Empty Abstracts: 0

Short Abstracts (<10 words): 5

Year Range: 2007 to 2021
Papers per Year:
year
2007      87
2008     131
2009     197
2010     310
2011     547
2012     600
2013    1455
2014     854
2015    1014
2016    1622
2017    2521
2018    2844
2019    2775
2020    4344
2021    7830
Name: count, dtype: int64

Cleaning data...
After dropping missing abstracts: 27131 papers. So removed 0
After dropping duplicate IDs: 27131 papers. So removed 0
After dropping duplicate abstracts: 27126 papers. So removed 5
After dropping empty abstracts: 27126 papers. So rem

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\11873\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!

100%|██████████| 27121/27121 [01:21<00:00, 332.71it/s]


In [3]:
import sys
# means: append the upper level folder (..) to this search list
sys.path.append('..')

In [4]:
import re
import numpy as np
import pandas as pd

In [5]:
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import PCA, TruncatedSVD, LatentDirichletAllocation
from sklearn.preprocessing import normalize

In [7]:
from tqdm import tqdm

In [8]:
df = pd.read_pickle("data_preprocessed.pkl")

In [9]:
print(f"Total number of filtered AI/ML/NLP papers: {len(df)}")

Total number of filtered AI/ML/NLP papers: 27121


In [10]:
display(df.head(3))

,id,submitter,authors,title,comments,journal-ref,doi,abstract,report-no,categories,versions,year,cleaned_abstract,period
0,0704.1274,Dev Rajnarayan,David H. Wolpert and Dev G. Rajnarayan,Parametric Learning and Monte Carlo Optimization,None,None,None,This paper uncovers and explores the close r...,None,[cs.LG],[v1],2007,paper uncovers explores close relationship mon...,2005-2009
1,0704.1394,Tarik Had\v{z}i\'c,"Tarik Hadzic, Rune Moller Jensen, Henrik Reif ...",Calculating Valid Domains for BDD-Based Intera...,None,None,None,In these notes we formally describe the func...,None,[cs.AI],[v1],2007,notes formally describe functionality calculat...,2005-2009
2,0704.2010,Juliana Bernardes,"Juliana S Bernardes, Alberto Davila, Vitor San...",A study of structural properties on profiles HMMs,"6 pages, 7 figures",None,None,Motivation: Profile hidden Markov Models (pH...,None,[cs.AI],"[v1, v2]",2007,motivation profile hidden markov models phmms ...,2005-2009


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [12]:
abstracts = df['cleaned_abstract'].dropna().tolist()
# Unigrams
# max_df=0.9: Kick out excessively common words that appear in more than 90% of articles
# min_df=5: Kick out rare words/typos that appear less than 5 times in 30,000 articles
# max_features=15: We only extract the 15 features with the highest overall weight/word frequency to see the effect
tfidf_uni = TfidfVectorizer(stop_words='english', ngram_range=(1, 1), max_df=0.9, min_df=5, max_features=15)
tfidf_uni.fit(abstracts)

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,analyzer,'word'
,stop_words,'english'
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"


In [13]:
print("The top 15 Unigrams with the highest frequency and weight:")
print(tfidf_uni.get_feature_names_out())

The top 15 Unigrams with the highest frequency and weight:
['approach' 'based' 'data' 'language' 'learning' 'model' 'models' 'neural'
 'paper' 'performance' 'propose' 'results' 'task' 'training' 'using']


In [14]:
# Bigrams
tfidf_bi = TfidfVectorizer(stop_words='english', ngram_range=(2, 2), max_df=0.9, min_df=5, max_features=15)
tfidf_bi.fit(abstracts)

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,analyzer,'word'
,stop_words,'english'
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(2, ...)"


In [15]:
print("The top 15 Bigrams with the highest frequency and weight")
print(tfidf_bi.get_feature_names_out())

The top 15 Bigrams with the highest frequency and weight
['deep learning' 'experimental results' 'language models'
 'language processing' 'machine learning' 'machine translation'
 'natural language' 'neural network' 'neural networks' 'paper propose'
 'pre trained' 'real world' 'reinforcement learning' 'state art'
 'training data']


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

# 1. Organize the useless words we found in the previous experiment
academic_boilerplate = {
    'paper', 'propose', 'proposed', 'approach', 'method', 'methods', 
    'results', 'experimental', 'experiment', 'show', 'shows', 
    'using', 'based', 'performance', 'task', 'state', 'art' 
}

# 2. Merge our custom vocabulary list with scikit-learn's built-in English stop vocabulary list (the, is, and...)
my_stop_words = list(ENGLISH_STOP_WORDS.union(academic_boilerplate))

# 3. Run the upgraded Biggrams extractor
tfidf_bi_clean = TfidfVectorizer(
    stop_words=my_stop_words, 
    ngram_range=(2, 2), 
    max_df=0.9, 
    min_df=5, 
    max_features=15
)

# Retraining
tfidf_bi_clean.fit(abstracts)

print("15 Bigrams after cutting out the academic nonsense:")
print(tfidf_bi_clean.get_feature_names_out())

15 Bigrams after cutting out the academic nonsense:
['deep learning' 'end end' 'language models' 'language processing'
 'large scale' 'machine learning' 'machine translation' 'natural language'
 'neural network' 'neural networks' 'pre trained' 'real world'
 'reinforcement learning' 'training data' 'word embeddings']


In [17]:
# Trigrams
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

#  Continue to use our prepared academic nonsense vocabulary list
academic_boilerplate = {
    'paper', 'propose', 'proposed', 'approach', 'method', 'methods', 
    'results', 'experimental', 'experiment', 'show', 'shows', 
    'using', 'based', 'performance', 'task', 'state', 'art' 
}
my_stop_words = list(ENGLISH_STOP_WORDS.union(academic_boilerplate))

# run the Trigrams extractor
tfidf_tri = TfidfVectorizer(
    stop_words=my_stop_words,
    ngram_range=(3,3),
    max_df=0.9,
    min_df=5,
    max_features=15)
tfidf_tri.fit(abstracts)

print("The top 15 Trigrams with the highest frequency and weight")
print(tfidf_tri.get_feature_names_out())




The top 15 Trigrams with the highest frequency and weight
['deep learning models' 'deep neural networks' 'language processing nlp'
 'long short term' 'machine learning models' 'machine translation nmt'
 'named entity recognition' 'natural language processing'
 'natural language understanding' 'neural machine translation'
 'pre trained language' 'recurrent neural network'
 'recurrent neural networks' 'short term memory' 'trained language models']


In [18]:
academic_boilerplate = {
    'paper', 'propose', 'proposed', 'approach', 'method', 'methods', 
    'results', 'experimental', 'experiment', 'show', 'shows', 
    'using', 'based', 'performance', 'task', 'state', 'art'
}

tfidf_final = TfidfVectorizer(
    stop_words=list(academic_boilerplate), 
    ngram_range=(2,2),
    max_df=0.9,
    min_df=5,
    max_features=5000
)

X_tfidf = tfidf_final.fit_transform(abstracts)


In [19]:
print(f"The shape of the matrix: {X_tfidf.shape}")
print("We have 30529 papers and extract the 5000 most core two-word features for them")

The shape of the matrix: (27121, 5000)
We have 30529 papers and extract the 5000 most core two-word features for them


In [20]:
feature_names_final = tfidf_final.get_feature_names_out()